In [22]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Probit
import pandas as pd
import numpy as np
from scipy.stats import norm
from statsmodels import stats




In [23]:
df = pd.read_csv("probit_dataset.csv")
df

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,mar_st,visit_doctor,work,alcohol,smoking,phys_active,region,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,142.0,0.0,0.0,0.0
4594,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,142.0,0.0,0.0,0.0
4595,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0
4596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0


In [24]:
df.columns

Index(['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology', 'age', 'income', 'n_child', 'sex',
       'type_area', 'invalid', 'mar_st', 'visit_doctor', 'work', 'alcohol',
       'smoking', 'phys_active', 'region', 'is_health_good',
       'is_health_very_good', 'diploma'],
      dtype='str')

In [25]:

# 1. Создаём словарь "код региона -> значение alcohol_by_region 
# (Потребление алкогольной продукции на душу населения (в литрах этанола) для этого региона)"
alcohol_dict = {
    1: 96.13,
    9: 77.48,
    10: 124.47,
    12: 121.17,
    14: 102.76,
    33: 93.26,
    39: 86.51,
    45: 100.86,
    46: 132.27,
    47: 96.62,
    48: 109.46,
    52: 49.65,
    58: 109.66,
    66: 100.53,
    67: 106.43,
    70: 100.44,
    71: 111.83,
    72: 109.08,
    73: 100.53,
    77: 28.97,
    84: 109.66,
    86: 85.72,
    89: 146.86,
    92: 99.02,
    93: 113.16,
    100: 100.44,
    105: 146.86,
    106: 120.05,
    107: 120.05,
    116: 103.08,
    117: 139.63,
    129: 77.48,
    135: 109.62,
    136: 100.89,
    137: 63.11,
    138: 60.22,
    141: 80.65,
    142: 97.76,
    161: 104.41,
    200: 117.44
}

# 2. Создаём словарь "код региона -> значение smoking_by_region 
# (Объем легальных розничных продаж сигарет на душу совершеннолетнего населения для этого региона)"
smoking_dict = {
    1: 411,
    9: 402,
    10: 307,
    12: 379,
    14: 344,
    33: 295,
    39: 278,
    45: 294,
    46: 377,
    47: 327,
    48: 275,      
    52: 248,
    58: 337,
    66: 361,
    67: 398,
    70: 311,
    71: 432,
    72: 313,
    73: 361,
    77: 80,
    84: 337,
    86: 432,      
    89: 539,      
    92: 483,
    93: 549,
    100: 311,
    105: 539,
    106: 354,
    107: 354,
    116: 384,     
    117: 300,
    129: 402,
    135: 300,
    136: 334,
    137: 301,
    138: 257,
    141: 283,
    142: 458,
    161: 323,
    200: 309
}

# 3. Создаём словарь "код региона -> значение marriages_by_region 
# (Число зарегистрированных браков в расчете на 1000 населения (оперативные данные) для этого региона)"

marriage_dict = {
    1: 3.9,
    9: 7.3,
    10: 5.4,
    12: 6.1,
    14: 5.3,
    33: 5.3,
    39: 5.5,
    45: 5.9,
    46: 5.9,
    47: 6.1,
    48: 4.4,
    52: 5.1,
    58: 6.5,
    66: 6.7,
    67: 6.0,
    70: 5.5,
    71: 6.6,
    72: 5.5,
    73: 6.7,
    77: 4.6,
    84: 6.5,
    86: 5.7,
    89: 5.9,
    92: 8.0,
    93: 7.7,
    100: 5.5,
    105: 5.9,
    106: 6.5,
    107: 6.5,
    116: 6.1,
    117: 5.4,
    129: 7.3,
    135: 5.8,
    136: 5.5,
    137: 6.0,
    138: 6.6,
    141: 9.0,
    142: 5.5,
    161: 7.1,
    200: 6.3
}


# 3. Создаём словарь "код региона -> значение phys_activity_by_region 
# (Рейтинговый балл по приверженности населения ЗОЖ для этого региона)"
phys_dict = {
    1: 60.2,
    9: 81.7,
    10: 55.2,
    12: 55.8,
    14: 72.7,
    33: 76.2,
    39: 75.7,
    45: 70.2,
    46: 59.3,
    47: 64.6,
    48: 74.8,      
    52: 81.5,
    58: 67.8,
    66: 46.7,
    67: 62.7,
    70: 68.9,
    71: 63.1,
    72: 72.5,
    73: 46.7,
    77: 81.8,
    84: 67.8,
    86: 58.9,      
    89: 54.6,      
    92: 56.9,
    93: 53.3,
    100: 68.9,
    105: 54.6,
    106: 43.2,
    107: 43.2,
    116: 66.2,     
    117: 76.0,
    129: 81.7,
    135: 69.6,
    136: 81.5,
    137: 74.9,
    138: 76.0,
    141: 79.3,
    142: 80.1,
    161: 60.9,
    200: 60.1
}


df['alcohol_by_region'] = df['region'].map(alcohol_dict)
df['smoking_by_region'] = df['region'].map(smoking_dict)
df['marriages_by_region'] = df['region'].map(marriage_dict)
df['phys_activity_by_region'] = df['region'].map(phys_dict)

df

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,smoking,phys_active,region,is_health_good,is_health_very_good,diploma,alcohol_by_region,smoking_by_region,marriages_by_region,phys_activity_by_region
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,96.13,411,3.9,60.2
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,96.13,411,3.9,60.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,142.0,0.0,0.0,0.0,97.76,458,5.5,80.1
4594,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,142.0,0.0,0.0,0.0,97.76,458,5.5,80.1
4595,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,77.0,0.0,0.0,0.0,28.97,80,4.6,81.8
4596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,77.0,0.0,0.0,0.0,28.97,80,4.6,81.8


In [26]:
df.columns

Index(['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology', 'age', 'income', 'n_child', 'sex',
       'type_area', 'invalid', 'mar_st', 'visit_doctor', 'work', 'alcohol',
       'smoking', 'phys_active', 'region', 'is_health_good',
       'is_health_very_good', 'diploma', 'alcohol_by_region',
       'smoking_by_region', 'marriages_by_region', 'phys_activity_by_region'],
      dtype='str')

In [27]:

def control_function_2spm(sex, illness, df, endog_vars, exog_vars, instruments):

    df_sex = df[df['sex'] == sex].copy()
    outcome = illness
    
    resid_cols = []
    
    for var in endog_vars:
        instr_for_var = instruments.get(var, [])
        first_step_vars = exog_vars + instr_for_var
        
        X1 = sm.add_constant(df_sex[first_step_vars])
        y1 = df_sex[var]
        
        probit1 = Probit(y1, X1)
        res1 = probit1.fit(disp=0)
        

        xb = res1.predict(X1, linear=True)
        prob = norm.cdf(xb)
        phi = norm.pdf(xb)
        
        generalized_resid = np.where(y1 == 1, phi/prob, -phi/(1-prob))
        resid_name = f'{var}_gresid'
        df_sex[resid_name] = generalized_resid
        resid_cols.append(resid_name)
    
    
    X2_vars = exog_vars + endog_vars + resid_cols
    X2 = sm.add_constant(df_sex[X2_vars])
    y2 = df_sex[outcome]
    

    
    probit2 = Probit(y2, X2)
    res2 = probit2.fit(disp=0)
    
    resid_cols_in_model = [col for col in resid_cols if col in res2.params.index]
    if resid_cols_in_model:
        
        # Wald тест
        constraints = [f'{col}=0' for col in resid_cols_in_model]
        wald_test = res2.wald_test(constraints)
        exogeneity_pval = wald_test.pvalue
    else:
        exogeneity_pval = np.nan
    
    print(res2.summary())

    return {
        'coef_diploma': res2.params.get('diploma', np.nan),
        'pval_diploma': res2.pvalues.get('diploma', np.nan),
        'exogeneity_pval': exogeneity_pval,
        'n_obs': len(y2),
        'pseudo_r2': res2.prsquared
    }


sexes = [1, 2]
illnesses = ['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology', 'is_health_good']

endog_vars = ['diploma', 'mar_st', 'alcohol', 'smoking', 'phys_active']
exog_vars = ['age', 'income', 'n_child', 'type_area', 'invalid', 'visit_doctor', 'work']
instruments = {
    'diploma': ['alcohol_by_region', 'smoking_by_region', 'marriages_by_region', 'phys_activity_by_region'],
    'mar_st': ['marriages_by_region'],
    'alcohol': ['alcohol_by_region'],
    'smoking': ['smoking_by_region'],
    'phys_active': ['phys_activity_by_region']
}

results = []
for sex in sexes:
    for illness in illnesses:
        res = control_function_2spm(sex, illness, df, endog_vars, exog_vars, instruments)
        results.append({'sex': sex, 'illness': illness, **res})

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: Futur

                          Probit Regression Results                           
Dep. Variable:                  heart   No. Observations:                 1926
Model:                         Probit   Df Residuals:                     1909
Method:                           MLE   Df Model:                           16
Date:                Mon, 04 May 2026   Pseudo R-squ.:                  0.1370
Time:                        14:18:24   Log-Likelihood:                -346.76
converged:                       True   LL-Null:                       -401.83
Covariance Type:            nonrobust   LLR p-value:                 4.237e-16
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
age                    0.0097      0.018      0.522      0.601      -0.027       0.046
income              5.378e-06    3.5e-06      1.538      0.124   -1.48e-06    1.22e-05
n_child             

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: Futur

                          Probit Regression Results                           
Dep. Variable:                  liver   No. Observations:                 1926
Model:                         Probit   Df Residuals:                     1909
Method:                           MLE   Df Model:                           16
Date:                Mon, 04 May 2026   Pseudo R-squ.:                 0.08908
Time:                        14:18:24   Log-Likelihood:                -264.93
converged:                       True   LL-Null:                       -290.84
Covariance Type:            nonrobust   LLR p-value:                 1.172e-05
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
age                    0.0327      0.023      1.451      0.147      -0.011       0.077
income               2.64e-06   3.82e-06      0.690      0.490   -4.86e-06    1.01e-05
n_child             

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: Futur

...

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.war

                          Probit Regression Results                           
Dep. Variable:                allergy   No. Observations:                 2672
Model:                         Probit   Df Residuals:                     2655
Method:                           MLE   Df Model:                           16
Date:                Mon, 04 May 2026   Pseudo R-squ.:                 0.03957
Time:                        14:18:27   Log-Likelihood:                -725.30
converged:                       True   LL-Null:                       -755.18
Covariance Type:            nonrobust   LLR p-value:                 5.744e-07
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
age                    0.0279      0.013      2.077      0.038       0.002       0.054
income              5.047e-06   5.67e-06      0.890      0.373   -6.06e-06    1.62e-05
n_child             

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.war

                          Probit Regression Results                           
Dep. Variable:                   skin   No. Observations:                 2672
Model:                         Probit   Df Residuals:                     2655
Method:                           MLE   Df Model:                           16
Date:                Mon, 04 May 2026   Pseudo R-squ.:                 0.03023
Time:                        14:18:27   Log-Likelihood:                -248.59
converged:                       True   LL-Null:                       -256.34
Covariance Type:            nonrobust   LLR p-value:                    0.4885
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
age                   -0.0086      0.022     -0.397      0.691      -0.051       0.034
income              -6.22e-06   1.06e-05     -0.588      0.557    -2.7e-05    1.45e-05
n_child             

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:530: FutureWarning: linear keyword is deprecated, use which="linear"
  warnings.warn(msg, FutureWarning)
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:1912: FutureWarning: Th

In [28]:
results

[{'sex': 1,
  'illness': 'heart',
  'coef_diploma': -0.9537989781474785,
  'pval_diploma': 0.4032938199458408,
  'exogeneity_pval': array(0.28812319),
  'n_obs': 1926,
  'pseudo_r2': 0.13703651729654331},
 {'sex': 1,
  'illness': 'lungs',
  'coef_diploma': -1.1566512217387324,
  'pval_diploma': 0.5295797315654724,
  'exogeneity_pval': array(0.01918627),
  'n_obs': 1926,
  'pseudo_r2': 0.061419987909364315},
 {'sex': 1,
  'illness': 'liver',
  'coef_diploma': -0.4687983572606453,
  'pval_diploma': 0.7193601332854489,
  'exogeneity_pval': array(0.99159377),
  'n_obs': 1926,
  'pseudo_r2': 0.08908324028510817},
 {'sex': 1,
  'illness': 'kidneys',
  'coef_diploma': -0.2569281583847186,
  'pval_diploma': 0.8528511245522301,
  'exogeneity_pval': array(0.89618589),
  'n_obs': 1926,
  'pseudo_r2': 0.0525254140583854},
 {'sex': 1,
  'illness': 'stomach',
  'coef_diploma': 0.6142737346018782,
  'pval_diploma': 0.5611516526832124,
  'exogeneity_pval': array(0.47411225),
  'n_obs': 1926,
  'pseudo